# Beam Deflection Verification — 3-Point Bending (RC & CRC)

Verifies the `BeamDeflectionAnalysis` BMCSModel introduced in
`scite/beam/bending/beam_deflection.py`.

**Scope**
- Section 1 — elastic verification against the analytical formula  
  $w_\mathrm{max} = \dfrac{F L^3}{48\, E_c I}$ for uncracked 3PB
- Section 2 — reference RC beam: nonlinear vs linear-elastic  
- Section 3 — span effect ($L$ sweep)
- Section 4 — reinforcement ratio effect ($A_s$ sweep)
- Section 5 — CRC beam with CFRP reinforcement

**Architecture note**  
`BeamDeflectionAnalysis` is a Pydantic `BMCSModel` (analogous to `MKappaAnalysis`).  
It takes a `CrossSection` directly; no UI framework is needed here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── scite cross-section design ────────────────────────────────────────────────
from scite.cs_design.shapes import RectangularShape, TShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout

# ── scite material models ─────────────────────────────────────────────────────
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import SteelReinforcement, create_steel
from scite.matmod.carbon_reinforcement import CarbonReinforcement, create_carbon
from scite.mkappa.mkappa import MKappaAnalysis
from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print("Imports OK")

---
## 1  Elastic verification — uncracked cross-section

For a linear-elastic simply supported beam under a midspan point load:

$$
w_\mathrm{max} = \frac{F L^3}{48\, E_c I}
$$

We verify this against `BeamDeflectionAnalysis` by using a **very low load**
(5 % of $F_R$), where the beam is still fully uncracked and the M-κ relationship
is linear.  The curvature at such a load is given by $\kappa = M / (E_c I)$, so
the integration should reproduce the elastic formula exactly.

In [ ]:
# ── Reference rectangular RC cross-section (200×400 mm, 2ø16 B500B) ──────────
b_ref, h_ref = 200.0, 400.0   # mm
A_s_ref      = 402.0           # mm²  (2ø16)
z_s_ref      = 45.0            # mm  (concrete cover + ø/2 from bottom)

concrete_ref = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)
steel        = create_steel('B500B')

shape_rect  = RectangularShape(b=b_ref, h=h_ref)
layer_ref   = ReinforcementLayer(material=steel, A_s=A_s_ref, z=z_s_ref)
reinf_ref   = ReinforcementLayout(layers=[layer_ref])

cs_rect = CrossSection(shape=shape_rect, concrete=concrete_ref, reinforcement=reinf_ref)
print('Cross-section: b=%g mm, h=%g mm, A_s=%g mm²' % (b_ref, h_ref, A_s_ref))

In [ ]:
L_ref = 6000.0   # mm

bda_ref = BeamDeflectionAnalysis(
    cs=cs_rect,
    L=L_ref,
    load_type='3pb',
    n_x=200,
    n_kappa=150,
    n_load_steps=41,
)

print(f'M_R = {bda_ref.M_R:.2f} kN·m')
print(f'F_R = {bda_ref.F_R/1000:.2f} kN  (= 4·M_R/L)')

In [ ]:
# ── Elastic verification: test integration accuracy against M-κ initial slope ─
F_el = 0.05 * bda_ref.F_R          # 5 % of F_R  — well below cracking
w_x  = bda_ref.get_w_x(F_el)
w_mid_bda = float(np.max(w_x))     # midspan deflection from BDA integration

# Extract effective bending stiffness EI from the M-κ slope (initial tangent)
mk    = bda_ref.mk
n_fit = min(10, len(mk.M_values))
EI_eff = float(np.polyfit(mk.kappa_values[1:n_fit],
                           mk.M_values[1:n_fit] * 1e6, 1)[0])  # N·mm²

# Analytical formula for simply supported beam, midspan point load
w_mid_ref = F_el * bda_ref.L**3 / (48.0 * EI_eff)

print('Elastic verification (integration accuracy check)')
print('─' * 55)
print(f'F_el              = {F_el:8.2f} N  ({100*F_el/bda_ref.F_R:.0f} % of F_R)')
print(f'EI_eff  (M-κ)     = {EI_eff:.3e} N·mm²  (from M-κ slope)')
print(f'E_cm·I_g (EC2)    = {bda_ref.E_cm * bda_ref.I_g:.3e} N·mm²  (mean props., for ref.)')
print(f'w_BDA (integr.)   = {w_mid_bda:.4f} mm')
print(f'w_elastic (M-κ)   = {w_mid_ref:.4f} mm  (analytical, same EI)')
print(f'Relative error    = {100*(w_mid_bda - w_mid_ref)/w_mid_ref:.2f} %  (integration accuracy)')
print()
print('Note: EI_eff < E_cm·I_g  because the M-κ model uses design concrete')
print('  (f_cd with γ_c = 1.5) and omits concrete tensile stiffness (ULS tool).')


In [ ]:
# ── Deflection and curvature profiles at several load levels ─────────────────
# Convention: w and κ shown downward (positive downward = sags below beam axis)
fig, (ax_w, ax_k) = plt.subplots(1, 2, figsize=(14, 4))
bda_ref.plot_profiles(
    ax_w=ax_w, ax_k=ax_k,
    fractions=(0.05, 0.30, 0.60, 1.00),
    show_elastic_ref=True,   # dotted elastic reference at 5 % F_R
    show_sls=True,
)
plt.tight_layout()
plt.show()


---
## 2  Nonlinear vs linear-elastic comparison

At service loads the cracked cross-section is significantly more flexible than
the uncracked gross-section assumption.  The linear-elastic formula with the
gross-section $I_g$ underestimates deflections; the nonlinear integration
captures the stiffness reduction after cracking.

In [ ]:
# ── Nonlinear vs linear-elastic comparison ───────────────────────────────────
# plot_Fw_vs_elastic() overlays the nonlinear BDA curve with
# the gross-section (I_g) and cracked-section (I_cr) elastic references
fig, ax = plt.subplots(figsize=(8, 5))
bda_ref.plot_Fw_vs_elastic(ax=ax)
ax.set_title(f'Nonlinear vs linear-elastic — RC {b_ref:.0f}×{h_ref:.0f} mm, L={L_ref/1000:.1f} m')
print(f'E_cm = {bda_ref.E_cm:.0f} MPa,  I_g = {bda_ref.I_g:.3e} mm⁴')
plt.tight_layout()
plt.show()


---
## 3  Summary plot

Three-panel overview using `BeamDeflectionAnalysis.plot_summary()`.

In [ ]:
bda_ref.plot_summary(
    title='RC 200×400 mm — B500B 2ø16 — L=6 m, 3PB',
)


---
## 4  Span effect — $L$ sweep

As the span increases:
- $F_R = 4 M_R / L$ decreases (longer beam, same moment capacity)
- But the SLS limit $L/250$ also increases linearly with $L$
- The key question is whether deflection at $F = \alpha \cdot F_R$ (e.g. 40 % of ULS)
  violates $L/250$

The nonlinear F-w curves are plotted in normalised form $w / L$ vs $F / F_R$
to compare slenderness effects.

In [ ]:
L_values = [3000.0, 4500.0, 6000.0, 8000.0, 10_000.0]   # mm
colors   = plt.cm.viridis(np.linspace(0.1, 0.9, len(L_values)))

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(14, 5))

for L, color in zip(L_values, colors):
    bda = BeamDeflectionAnalysis(cs=cs_rect, L=L, load_type='3pb', n_load_steps=41)
    F_arr, w_arr = bda.get_Fw()
    lbl = f'L = {L/1000:.1f} m'
    ax_abs.plot(w_arr, F_arr/1e3, color=color, lw=1.8, label=lbl)
    ax_norm.plot(w_arr/L * 1e3, F_arr/bda.F_R, color=color, lw=1.8, label=lbl)

ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]'); ax_abs.set_ylabel('F [kN]')
ax_abs.set_title('Absolute F-w'); ax_abs.legend(fontsize=8); ax_abs.grid(True, alpha=0.3)

ax_norm.axvline(1/250*1e3, color='red', linestyle='--', lw=1.2, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max} / L$ [‰]'); ax_norm.set_ylabel(r'$F / F_R$')
ax_norm.set_title('Normalised F-w')
ax_norm.legend(fontsize=8); ax_norm.grid(True, alpha=0.3)

plt.suptitle('Span sweep — RC 200×400, B500B 2ø16', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 5  Reinforcement ratio — $A_s$ sweep

Increasing $A_s$ raises the moment capacity $M_R$ and reduces deflections at
the same load level.  The normalised plot reveals the diminishing return of
additional reinforcement once the concrete-governed failure mode is approached.

In [ ]:
A_s_values = [100.0, 200.0, 402.0, 600.0, 1000.0, 1600.0]   # mm²
colors = plt.cm.plasma(np.linspace(0.1, 0.85, len(A_s_values)))

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(14, 5))

for A_s, color in zip(A_s_values, colors):
    layer = ReinforcementLayer(material=steel, A_s=A_s, z=z_s_ref)
    cs    = CrossSection(shape=shape_rect, concrete=concrete_ref,
                         reinforcement=ReinforcementLayout(layers=[layer]))
    bda   = BeamDeflectionAnalysis(cs=cs, L=L_ref, load_type='3pb', n_load_steps=41)
    F_arr, w_arr = bda.get_Fw()
    lbl = f'A_s = {A_s:.0f} mm²'
    ax_abs.plot(w_arr, F_arr/1e3, color=color, lw=1.8, label=lbl)
    ax_norm.plot(w_arr/L_ref * 1e3, F_arr/bda.F_R, color=color, lw=1.8, label=lbl)
    fm_as = bda.mk.get_failure_mode()
    print(f'A_s={A_s:6.0f} mm²  M_R={bda.M_R:6.1f} kN·m  mode={fm_as["mode"]:16s}  η_c={fm_as["eta_c"]:.3f}  η_f={fm_as["eta_f"]:.3f}')

ax_abs.axvline(L_ref/250, color='red', linestyle='--', lw=1.2, label=f'L/250={L_ref/250:.0f} mm')
ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]'); ax_abs.set_ylabel('F [kN]')
ax_abs.set_title('Absolute F-w'); ax_abs.legend(fontsize=8); ax_abs.grid(True, alpha=0.3)

ax_norm.axvline(1/250*1e3, color='red', linestyle='--', lw=1.2, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max}/L$ [‰]'); ax_norm.set_ylabel(r'$F/F_R$')
ax_norm.set_title('Normalised F-w')
ax_norm.legend(fontsize=8); ax_norm.grid(True, alpha=0.3)

plt.suptitle(r'$A_s$ sweep — RC 200×400, B500B, L=6 m', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 6  CRC beam with CFRP reinforcement

The CRC beam (Carbon-Reinforced Concrete) uses a thin-walled T-section with
CFRP grid reinforcement.  Key differences from conventional RC:

- **Linear-elastic until rupture** — no yielding plateau
- Much lower reinforcement area, very high tensile strength
- Reinforcement-governed failure (η_f ≈ 1)
- Higher stiffness per unit weight

In [ ]:
# ── CRC T-section — Solidian Q95 CFRP grid ───────────────────────────────────
b_w_crc  = 50.0;  h_w_crc = 160.0   # web  [mm]
b_f_crc  = 200.0; h_f_crc = 40.0    # flange [mm]
z_reinf  = 20.0                      # reinforcement layer from bottom [mm]
A_s_crc  = 100.5                     # mm²  (Solidian Q95 per 200 mm width)

concrete_crc = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)

# Solidian Q95 carbon grid: f_t = 3000 MPa, E = 210 GPa
cfrp = CarbonReinforcement(
    E=210_000.0,   # MPa  (210 GPa)
    f_t=3000.0,    # MPa
)
print(f'CFRP (Solidian Q95): E = {cfrp.E/1000:.0f} GPa,  f_t = {cfrp.f_t:.0f} MPa,  '
      f'ε_cr = {cfrp.f_t/cfrp.E*100:.2f} %')

shape_T_crc = TShape(b_w=b_w_crc, h_w=h_w_crc, b_f=b_f_crc, h_f=h_f_crc)
layer_crc   = ReinforcementLayer(material=cfrp, A_s=A_s_crc, z=z_reinf)
cs_crc      = CrossSection(shape=shape_T_crc, concrete=concrete_crc,
                            reinforcement=ReinforcementLayout(layers=[layer_crc]))

bda_crc = BeamDeflectionAnalysis(cs=cs_crc, L=L_ref, load_type='3pb', n_load_steps=41)
print(f'CRC beam: M_R = {bda_crc.M_R:.2f} kN·m,  F_R = {bda_crc.F_R/1e3:.2f} kN')

fm_crc = bda_crc.mk.get_failure_mode()
print(f'Failure mode: {fm_crc["mode"]:16s}  η_c={fm_crc["eta_c"]:.3f}  η_f={fm_crc["eta_f"]:.3f}')


In [ ]:
# ── RC T-section — same geometry and reinforcement area, steel B500B ─────────
# Same shape_T_crc, same z_reinf, same A_s_crc → initial stiffness dominated
# by concrete cross-section, so uncracked stiffness is essentially identical.
layer_rc = ReinforcementLayer(material=steel, A_s=A_s_crc, z=z_reinf)
cs_rc    = CrossSection(shape=shape_T_crc, concrete=concrete_crc,
                         reinforcement=ReinforcementLayout(layers=[layer_rc]))

bda_rc = BeamDeflectionAnalysis(cs=cs_rc, L=L_ref, load_type='3pb', n_load_steps=41)
print(f'RC  beam: M_R = {bda_rc.M_R:.2f} kN·m,  F_R = {bda_rc.F_R/1e3:.2f} kN')
fm_rc = bda_rc.mk.get_failure_mode()
print(f'Failure mode: {fm_rc["mode"]:16s}  η_c={fm_rc["eta_c"]:.3f}  η_f={fm_rc["eta_f"]:.3f}')

fig, (ax_abs, ax_norm) = plt.subplots(1, 2, figsize=(14, 5))

F_arr_crc, w_arr_crc = bda_crc.get_Fw()
F_arr_rc,  w_arr_rc  = bda_rc.get_Fw()

ax_abs.plot(w_arr_crc, F_arr_crc/1e3, color='#2ca02c', lw=2.2,
            label=f'CRC — CFRP (Solidian Q95)  M_R={bda_crc.M_R:.1f} kN·m')
ax_abs.plot(w_arr_rc,  F_arr_rc/1e3,  color='#1f77b4', lw=2.2, linestyle='--',
            label=f'RC  — B500B steel          M_R={bda_rc.M_R:.1f} kN·m')
ax_abs.axvline(L_ref/250, color='red', linestyle='--', lw=1.2, label=f'L/250={L_ref/250:.0f} mm')
ax_abs.set_xlabel(r'$w_\mathrm{max}$ [mm]'); ax_abs.set_ylabel('F [kN]')
ax_abs.set_title('CRC vs RC — absolute F-w'); ax_abs.legend(fontsize=8); ax_abs.grid(True, alpha=0.3)

ax_norm.plot(w_arr_crc/L_ref*1e3, F_arr_crc/bda_crc.F_R, color='#2ca02c', lw=2.2, label='CRC (CFRP)')
ax_norm.plot(w_arr_rc/ L_ref*1e3, F_arr_rc /bda_rc.F_R,  color='#1f77b4', lw=2.2, linestyle='--', label='RC (B500B)')
ax_norm.axvline(1/250*1e3, color='red', linestyle='--', lw=1.2, label='L/250')
ax_norm.set_xlabel(r'$w_\mathrm{max}/L$ [‰]'); ax_norm.set_ylabel(r'$F/F_R$')
ax_norm.set_title('Normalised — CRC vs RC')
ax_norm.legend(fontsize=8); ax_norm.grid(True, alpha=0.3)

plt.suptitle(f'CRC vs RC — T-section, A_s={A_s_crc:.1f} mm², L=6 m, 3PB', fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
bda_crc.plot_summary(
    title='CRC T-section — CFRP 100.5 mm² — L=6 m, 3PB',
)


---
## 7  Distributed load — `load_type='dist'`

Verify the uniform distributed load case against the elastic formula
$w_\mathrm{max} = 5 q L^4 / (384\, E_c I)$.

In [ ]:
bda_dist = BeamDeflectionAnalysis(
    cs=cs_rect, L=L_ref, load_type='dist', n_x=200, n_kappa=150, n_load_steps=41,
)
print(f'dist load: M_R = {bda_dist.M_R:.2f} kN·m')
print(f'  q_R = {bda_dist.F_R:.4f} N/mm  = {bda_dist.F_R:.2f} kN/m')

# Integration accuracy check: extract EI_eff from M-κ slope
mk_d   = bda_dist.mk
n_fit  = min(10, len(mk_d.M_values))
EI_eff = float(np.polyfit(mk_d.kappa_values[1:n_fit],
                           mk_d.M_values[1:n_fit] * 1e6, 1)[0])  # N·mm²

q_el           = 0.05 * bda_dist.F_R
w_dist_bda     = float(np.max(bda_dist.get_w_x(q_el)))
w_dist_elastic = 5 * q_el * L_ref**4 / (384 * EI_eff)

print(f'  q_el = {q_el:.4f} N/mm  (5 % of q_R)')
print(f'  w_BDA = {w_dist_bda:.4f} mm  (numerical integration)')
print(f'  w_el  = {w_dist_elastic:.4f} mm  (analytical, same EI_eff)')
print(f'  Relative error = {100*(w_dist_bda - w_dist_elastic)/w_dist_elastic:.2f} %  (integration accuracy)')

# F-w curve
F_arr_dist, w_arr_dist = bda_dist.get_Fw()
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(w_arr_dist, F_arr_dist, color='darkorange', lw=2.0, label='Nonlinear — dist load')
ax.axvline(L_ref/250, color='red', linestyle='--', lw=1.2, label=f'L/250={L_ref/250:.0f} mm')
ax.set_xlabel(r'$w_\mathrm{max}$ [mm]', fontsize=11)
ax.set_ylabel(r'$q$ [N/mm]', fontsize=11)
ax.set_title(f'Uniform distributed load — RC 200×400, L=6 m')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
